# **DATA BALANCING NOTEBOOK**

## **0. Library import**

This section imports the libraries and modules needed for data balancing. This includes data processing libraries such as pandas, sklearn for data splitting, and custom modules for feature engineering and data balancing.

In [1]:
import os
import sys

# Add the root path into the python path
root_path = os.path.abspath(os.path.join(".."))
if not root_path in sys.path:
    sys.path.insert(0, root_path)

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from src.utils import save_data
from src.config import BRFSS_CLEANED_FILE_PATH, LOG_DIR, TRAINING_DIR, \
    RANDOM_OVER_SAMPLING_DATA_FILE_PATH, SMOTE_DATA_FILE_PATH, ADASYN_DATA_FILE_PATH, \
    RANDOM_UNDER_SAMPLING_DATA_FILE_PATH, TOMEK_LINKS_DATA_FILE_PATH, EDITED_NEIGHBORS_DATA_FILE_PATH, \
    SMOTE_TOMEK_LINKS_DATA_FILE_PATH, SMOTE_ENN_DATA_FILE_PATH, ADASYN_TOMEK_DATA_FILE_PATH, \
    TESTING_DATA_FILE_PATH, TRAIN_SIZE, N_JOBS, OVER_SAMPLING_STRATEGY, UNDER_SAMPLING_STRATEGY
from src.features import DiabetesFeatureEngineering
from src.balancing import OverSamplingBalancer, UnderSamplingBalancer, HybridSamplingBalancer

## **1. Load data**

Load preprocessed data from a CSV file. This data has undergone filtering and cleaning from previous steps, containing information about health indicators and the Diabetes target variable with three classes: No diabetes (0), Pre-diabetes (1), and Diabetes (2).

In [3]:
# Load preprocessed BRFSS dataset for data balancing
# Contains imbalanced classes: No diabetes (81.8%), Pre-diabetes (2.4%), Diabetes (15.8%)
df = pd.read_csv(filepath_or_buffer=BRFSS_CLEANED_FILE_PATH)
df.head()

,Diabetes,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,HvyAlcoholConsump,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Year
0,2.0,1.0,1.0,1.0,27.0,0.0,0.0,0.0,1.0,0.0,...,0.0,2.0,0.0,0.0,1.0,0.0,11.0,6.0,6.0,2017
1,0.0,1.0,0.0,1.0,29.0,0.0,0.0,0.0,1.0,0.0,...,0.0,2.0,0.0,0.0,0.0,1.0,10.0,6.0,8.0,2017
2,0.0,0.0,0.0,1.0,23.0,1.0,0.0,0.0,0.0,0.0,...,0.0,4.0,0.0,14.0,0.0,0.0,10.0,2.0,2.0,2017
3,0.0,1.0,0.0,1.0,27.0,1.0,0.0,1.0,1.0,0.0,...,0.0,3.0,0.0,6.0,0.0,1.0,12.0,4.0,4.0,2017
4,0.0,0.0,0.0,1.0,28.0,0.0,0.0,0.0,0.0,0.0,...,0.0,3.0,0.0,0.0,0.0,1.0,10.0,5.0,8.0,2017


## **2. Feature engineering**

Apply feature engineering techniques to create new features from the original data to improve model performance. This process includes creating composite metrics such as Health Score, Risk Score, and categorical variables such as BMI Category, Age Group. The goal is to create highly informative features that help the machine learning model recognize important patterns in the data.

Generate composite indices from multiple related variables:

- **Health Score**: Composite health index based on GenHlth, MentHlth, PhysHlth
- **Risk Score**: Risk score based on HighBP, HighChol, HeartDiseaseorAttack, Stroke
- **Lifestyle Score**: Lifestyle score based on PhysActivity and negative factors such as HvyAlcoholConsump, Smoker
- **Cardio Risk**: Cardiovascular risk based on HighBP, HighChol and BMI obesity

Convert continuous variables into meaningful categories:

- **BMI Category**: BMI classification according to WHO standards (Underweight, Normal, Pre-obesity, Obesity class I-III)
- **Age Group**: Age grouping into meaningful ranges (Young, Middle-aged, Senior, Elderly)


In [4]:
# Apply a complete feature engineering pipeline:
# - Remove duplicates and handle outliers
# - Create composite scores (Health, Risk, Lifestyle, Cardio)
# - Feature selection based on a correlation threshold (>0.1)
diabetes_feature_engineering = DiabetesFeatureEngineering(log_file=f"{LOG_DIR}/4_feature_engineering_pipeline.log")
processed_df, encoders, selected_features = diabetes_feature_engineering.process_all(df=df)
processed_df.shape

2025-09-17 20:12:01,232 - [src.features] - INFO - DiabetesFeatureEngineering initialized successfully
2025-09-17 20:12:01,234 - [src.features] - INFO - ============================================================
2025-09-17 20:12:01,235 - [src.features] - INFO - STARTING COMPLETE FEATURE ENGINEERING PIPELINE
2025-09-17 20:12:01,236 - [src.features] - INFO - ============================================================
2025-09-17 20:12:01,237 - [src.features] - INFO - Initial dataset shape: (787602, 21)
2025-09-17 20:12:01,237 - [src.features] - INFO - Starting null values removal process...
2025-09-17 20:12:01,273 - [src.features] - INFO - No null values found in the dataset
2025-09-17 20:12:01,371 - [src.features] - INFO - Null values removal completed. Removed 0 rows (0.00%)
2025-09-17 20:12:01,372 - [src.features] - INFO - Dataset shape: 787602 -> 787602 rows
2025-09-17 20:12:01,373 - [src.features] - INFO - After null removal: (787602, 21)
2025-09-17 20:12:01,375 - [src.features] - 

(702516, 15)

In [5]:
# Save selected features
save_data(
    path=f"{TRAINING_DIR}/selected_features.pkl", 
    data=selected_features
)
# Save encoders
save_data(
    path=f"{TRAINING_DIR}/encoders.pkl", 
    data=encoders
)

In [6]:
# Display the original class imbalance before balancing
# Class 0 (No diabetes): ~82% | Class 1 (Pre-diabetes): ~2% | Class 2 (Diabetes):
processed_df["Diabetes"].value_counts()

Diabetes
0.0    574442
2.0    111184
1.0     16890
Name: count, dtype: int64

In [7]:
# Get features and target
X = processed_df.drop(columns=["Diabetes"])
y = processed_df["Diabetes"]

In [8]:
# Split training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    train_size=TRAIN_SIZE, 
    random_state=42, 
    stratify=y
)

In [9]:
# Export testing data
save_data(
    path=TESTING_DATA_FILE_PATH,
    data={
        "X": X_test,
        "y": y_test
    }
)

## **3. Data balancing**

Addressing the data imbalance issue in the BRFSS dataset, where the "No diabetes" class is the majority (459,553 samples) compared to "Pre-diabetes" (13,512 samples) and "Diabetes" (88,947 samples). Data balance is extremely important to ensure that the model is not biased towards the majority class and can predict accurately for all classes.

### **3.1 Oversampling**

In [10]:
# Initialize OverSamplingBalancer
over_sampling_balancer = OverSamplingBalancer(log_file=f"{LOG_DIR}/4_over_sampling_balancer.log")

2025-09-17 20:12:08,659 - [src.balancing] - INFO - OverSamplingBalancer initialized successfully


#### **3.1.1 Random Oversampling**

Random Oversampling is the simplest oversampling technique that addresses class imbalance by randomly duplicating existing samples from minority classes until the desired class distribution is achieved. This method replicates exact copies of minority class samples without creating new synthetic data.

Key characteristics:
- **Simplicity**: Easy to implement and understand, requires no complex algorithms
- **Speed**: Fastest oversampling method as it only involves copying existing data
- **Preservation**: Maintains exact original data points without modification
- **Risk of Overfitting**: May cause models to memorize duplicated samples rather than learn generalizable patterns

For the diabetes dataset, Random Oversampling increases the minority classes (Pre-diabetes and Diabetes) by duplicating existing patient records. While this ensures balanced class representation, the duplicated samples may lead to overfitting, especially for tree-based models that can easily memorize repeated patterns.

In [11]:
# Apply Random Oversampling method with sampling strategy {1.0: 50000,2.0: 100000} on training data
ros_X_train, ros_y_train = over_sampling_balancer.apply_random_oversampling(
    X=X_train, 
    y=y_train, 
    sampling_strategy=OVER_SAMPLING_STRATEGY
)

2025-09-17 20:12:08,677 - [src.balancing] - INFO - Starting Random Over Sampling process...
2025-09-17 20:12:08,693 - [src.balancing] - INFO - Original class distribution:
2025-09-17 20:12:08,695 - [src.balancing] - INFO -   - Class 0.0: 459553 samples
2025-09-17 20:12:08,696 - [src.balancing] - INFO -   - Class 1.0: 13512 samples
2025-09-17 20:12:08,697 - [src.balancing] - INFO -   - Class 2.0: 88947 samples
2025-09-17 20:12:08,916 - [src.balancing] - INFO - New class distribution after Random Over Sampling:
2025-09-17 20:12:08,918 - [src.balancing] - INFO -   - Class 0.0: 459553 samples (+0)
2025-09-17 20:12:08,919 - [src.balancing] - INFO -   - Class 1.0: 50000 samples (+36488)
2025-09-17 20:12:08,920 - [src.balancing] - INFO -   - Class 2.0: 150000 samples (+61053)
2025-09-17 20:12:08,921 - [src.balancing] - INFO - Dataset size: 562012 -> 659553 samples (+97541)
2025-09-17 20:12:08,922 - [src.balancing] - INFO - Random Over Sampling completed successfully


In [12]:
# Verify balanced distribution after resampling
ros_y_train.value_counts()

Diabetes
0.0    459553
2.0    150000
1.0     50000
Name: count, dtype: int64

In [13]:
# Save Random Oversampling balanced dataset for model training
save_data(
    path=RANDOM_OVER_SAMPLING_DATA_FILE_PATH,
    data={
        "X": ros_X_train,
        "y": ros_y_train,
    }
)

#### **3.1.2 SMOTE**

SMOTE (Synthetic Minority Oversampling Technique) generates synthetic samples by interpolating between existing minority class samples and their k-nearest neighbors. Instead of duplicating existing samples, SMOTE creates new, realistic samples that lie along the line segments connecting minority class samples.

Algorithm process:
1. **Neighbor Selection**: For each minority sample, find k-nearest neighbors from the same class
2. **Synthetic Generation**: Create new samples along the line connecting the sample to randomly selected neighbors
3. **Feature Interpolation**: New sample features = original + random_factor × (neighbor - original)

Advantages for medical data:
- **Diversity**: Creates varied synthetic patients with realistic feature combinations
- **Continuity**: Maintains data distribution characteristics while expanding minority regions
- **Reduced Overfitting**: Synthetic samples provide more generalizable training examples
- **Clinical Relevance**: Generated patients represent plausible health profiles between existing cases

SMOTE is particularly effective for diabetes prediction as it creates realistic patient profiles that help models learn decision boundaries between healthy, pre-diabetic, and diabetic populations.

In [14]:
# Apply SMOTE method with sampling strategy {1.0: 50000,2.0: 100000} on training data
smote_X_train, smote_y_train = over_sampling_balancer.apply_smote(
    X=X_train, 
    y=y_train, 
    sampling_strategy=OVER_SAMPLING_STRATEGY
)

2025-09-17 20:12:09,239 - [src.balancing] - INFO - Starting SMOTE process...
2025-09-17 20:12:09,253 - [src.balancing] - INFO - Original class distribution:
2025-09-17 20:12:09,254 - [src.balancing] - INFO -   - Class 0.0: 459553 samples
2025-09-17 20:12:09,255 - [src.balancing] - INFO -   - Class 1.0: 13512 samples
2025-09-17 20:12:09,256 - [src.balancing] - INFO -   - Class 2.0: 88947 samples
2025-09-17 20:12:23,590 - [src.balancing] - INFO - New class distribution after SMOTE:
2025-09-17 20:12:23,591 - [src.balancing] - INFO -   - Class 0.0: 459553 samples (+0 synthetic)
2025-09-17 20:12:23,592 - [src.balancing] - INFO -   - Class 1.0: 50000 samples (+36488 synthetic)
2025-09-17 20:12:23,593 - [src.balancing] - INFO -   - Class 2.0: 150000 samples (+61053 synthetic)
2025-09-17 20:12:23,594 - [src.balancing] - INFO - Dataset size: 562012 -> 659553 samples (+97541 synthetic)
2025-09-17 20:12:23,596 - [src.balancing] - INFO - SMOTE completed successfully


In [15]:
# Verify balanced distribution after resampling
smote_y_train.value_counts()

Diabetes
0.0    459553
2.0    150000
1.0     50000
Name: count, dtype: int64

In [16]:
# Save SMOTE balanced dataset for model training
save_data(
    path=SMOTE_DATA_FILE_PATH,
    data={
        "X": smote_X_train,
        "y": smote_y_train,
    }
)

#### **3.1.3 ADASYN**

ADASYN (ADAptive SYNthetic sampling) is an advanced oversampling technique that improves upon SMOTE by adaptively generating more synthetic samples in regions where minority class samples are harder to learn. Unlike SMOTE's uniform generation, ADASYN focuses on difficult minority class regions.

Key improvements over SMOTE:
- **Adaptive Density**: Generates more synthetic samples in sparse minority regions
- **Learning Difficulty Assessment**: Identifies hard-to-classify minority samples based on their neighborhood
- **Focused Generation**: Concentrates synthetic sample creation where it's most needed
- **Better Boundary Definition**: Improves class boundaries in challenging regions

Medical relevance:
- **Edge Case Coverage**: Creates more samples for rare but important diabetes presentations
- **Diagnostic Accuracy**: Helps models better distinguish between borderline cases
- **Clinical Diversity**: Ensures representation of uncommon but medically significant patterns
- **Reduced Misclassification**: Particularly effective for catching pre-diabetes cases that might otherwise be missed

ADASYN is especially valuable for diabetes prediction where identifying pre-diabetes (the smallest class) is crucial for early intervention and preventing progression to full diabetes.

In [17]:
# Apply ADASYN method with sampling strategy {1.0: 50000,2.0: 100000} on training data
adasyn_X_train, adasyn_y_train = over_sampling_balancer.apply_adasyn(
    X=X_train, 
    y=y_train, 
    sampling_strategy=OVER_SAMPLING_STRATEGY
)

2025-09-17 20:12:23,861 - [src.balancing] - INFO - Starting ADASYN over-sampling process...
2025-09-17 20:12:23,875 - [src.balancing] - INFO - Original class distribution:
2025-09-17 20:12:23,876 - [src.balancing] - INFO -  - Class 0.0: 459553 samples
2025-09-17 20:12:23,877 - [src.balancing] - INFO -  - Class 1.0: 13512 samples
2025-09-17 20:12:23,878 - [src.balancing] - INFO -  - Class 2.0: 88947 samples
2025-09-17 20:13:09,983 - [src.balancing] - INFO - New class distribution after ADASYN:
2025-09-17 20:13:09,984 - [src.balancing] - INFO -  - Class 0.0: 459553 samples (+0 synthetic)
2025-09-17 20:13:09,986 - [src.balancing] - INFO -  - Class 1.0: 50523 samples (+37011 synthetic)
2025-09-17 20:13:09,987 - [src.balancing] - INFO -  - Class 2.0: 147562 samples (+58615 synthetic)
2025-09-17 20:13:09,989 - [src.balancing] - INFO - Dataset size: 562012 -> 657638 samples (+95626 synthetic)
2025-09-17 20:13:09,991 - [src.balancing] - INFO - ADASYN over-sampling completed successfully


In [18]:
# Verify balanced distribution after resampling
adasyn_y_train.value_counts()

Diabetes
0.0    459553
2.0    147562
1.0     50523
Name: count, dtype: int64

In [19]:
# Save ADASYN balanced dataset for model training
save_data(
    path=ADASYN_DATA_FILE_PATH,
    data={
        "X": adasyn_X_train,
        "y": adasyn_y_train,
    }
)

### **3.2 Undersampling**

The method reduces the number of samples of the majority class to balance with the minority classes, which reduces training time but may lose important information.

In [20]:
# Initialize UnderSamplingBalancer
under_sampling_balancer = UnderSamplingBalancer(log_file=f"{LOG_DIR}/4_under_sampling_balancer.log")

2025-09-17 20:13:10,271 - [src.balancing] - INFO - UnderSamplingBalancer initialized successfully


#### **3.2.1 Random Undersampling**

Random Undersampling addresses class imbalance by randomly removing samples from the majority class until the desired class distribution is achieved. This method reduces the dataset size by discarding majority class samples without considering their informational value.

Characteristics:
- **Speed**: Fastest method for creating balanced datasets due to simple random selection
- **Memory Efficiency**: Significantly reduces dataset size, requiring less computational resources
- **Information Loss**: May discard important majority class patterns and edge cases
- **Risk of Underfitting**: Reduced dataset size might not provide enough information for complex model training

For diabetes prediction:
- **Computational Benefits**: Ideal when working with limited computational resources or time constraints
- **Population Sampling**: Can be viewed as creating a random sample of the non-diabetic population
- **Caution Required**: Risk of losing important patterns in healthy population that distinguish them from at-risk individuals

The method reduces the "No diabetes" class from ~460K to match minority class sizes, creating a more manageable dataset but potentially losing valuable information about the healthy population's diversity.

In [21]:
# Apply Random Undersampling method with {0: 100000, 2: 100000, 1: 16890} on training data
rus_X_train, rus_y_train = under_sampling_balancer.apply_random_undersampling(
    X=X_train, 
    y=y_train,
    sampling_strategy=UNDER_SAMPLING_STRATEGY
)

2025-09-17 20:13:10,293 - [src.balancing] - INFO - Starting Random Under Sampling process...
2025-09-17 20:13:10,307 - [src.balancing] - INFO - Original class distribution:
2025-09-17 20:13:10,309 - [src.balancing] - INFO -   - Class 0.0: 459553 samples
2025-09-17 20:13:10,310 - [src.balancing] - INFO -   - Class 1.0: 13512 samples
2025-09-17 20:13:10,312 - [src.balancing] - INFO -   - Class 2.0: 88947 samples
2025-09-17 20:13:10,494 - [src.balancing] - INFO - New class distribution after Random Under Sampling:
2025-09-17 20:13:10,495 - [src.balancing] - INFO -   - Class 0.0: 100000 samples (-359553)
2025-09-17 20:13:10,496 - [src.balancing] - INFO -   - Class 1.0: 13512 samples (-0)
2025-09-17 20:13:10,497 - [src.balancing] - INFO -   - Class 2.0: 88947 samples (-0)
2025-09-17 20:13:10,498 - [src.balancing] - INFO - Dataset size: 562012 -> 202459 samples (-359553)
2025-09-17 20:13:10,499 - [src.balancing] - INFO - Random Under Sampling completed successfully


In [22]:
# Verify balanced distribution after resampling
rus_y_train.value_counts()

Diabetes
0.0    100000
2.0     88947
1.0     13512
Name: count, dtype: int64

In [23]:
# Save Random Undersamplinng balanced dataset for model training
save_data(
    path=RANDOM_UNDER_SAMPLING_DATA_FILE_PATH,
    data={
        "X": rus_X_train,
        "y": rus_y_train,
    }
)

#### **3.2.2 TomekLinks**

The under-sampling method is smarter, only discarding the majority class samples that are close to the boundary decision and may cause noise. TomekLinks identifies pairs of samples from different classes that are nearest neighbors to each other and discards the majority class sample in that pair. Results: 444,228 (Class 0), 13,512 (Class 1), 74,901 (Class 2)—retains more information than random undersampling

In [24]:
# Apply Tomek Links method with sampling strategy 'auto' on training data
tomek_X_train, tomek_y_train = under_sampling_balancer.apply_tomek_links(
    X=X_train, 
    y=y_train, 
    n_jobs=N_JOBS
)

2025-09-17 20:13:10,648 - [src.balancing] - INFO - Starting Tomek Links process...
2025-09-17 20:13:10,666 - [src.balancing] - INFO - Original class distribution:
2025-09-17 20:13:10,669 - [src.balancing] - INFO -   - Class 0.0: 459553 samples
2025-09-17 20:13:10,671 - [src.balancing] - INFO -   - Class 1.0: 13512 samples
2025-09-17 20:13:10,673 - [src.balancing] - INFO -   - Class 2.0: 88947 samples
2025-09-17 20:14:05,814 - [src.balancing] - INFO - New class distribution after Tomek Links:
2025-09-17 20:14:05,815 - [src.balancing] - INFO -   - Class 0.0: 450210 samples (-9343 Tomek links)
2025-09-17 20:14:05,816 - [src.balancing] - INFO -   - Class 1.0: 13512 samples (unchanged)
2025-09-17 20:14:05,817 - [src.balancing] - INFO -   - Class 2.0: 79599 samples (-9348 Tomek links)
2025-09-17 20:14:05,819 - [src.balancing] - INFO - Dataset size: 562012 -> 543321 samples (-18691 Tomek links)
2025-09-17 20:14:05,820 - [src.balancing] - INFO - Tomek Links processing completed successfully


In [25]:
# Verify balanced distribution after resampling
tomek_y_train.value_counts()

Diabetes
0.0    450210
2.0     79599
1.0     13512
Name: count, dtype: int64

In [26]:
# Save Tomek Links balanced dataset for model training
save_data(
    path=TOMEK_LINKS_DATA_FILE_PATH,
    data={
        "X": tomek_X_train,
        "y": tomek_y_train,
    }
)

#### **3.2.3 Edited Nearest Neighbors**

Edited Nearest Neighbors (ENN) is an intelligent under-sampling technique that removes samples whose class differs from the majority of their k-nearest neighbors. Unlike random undersampling, ENN focuses on cleaning noisy and mislabeled samples from all classes, not just the majority class.

Key characteristics of ENN:
- **Noise Reduction**: Removes samples that are likely mislabeled or in noisy regions
- **Boundary Cleaning**: Improves class boundaries by removing ambiguous samples
- **Multi-class Approach**: Works on all classes simultaneously, not just majority class
- **Quality over Quantity**: Prioritizes data quality over maintaining large sample sizes

The method is particularly effective for medical data where accurate diagnosis is crucial and noisy samples could lead to misclassification in critical cases.

In [27]:
# Apply Edited Nearest Neighbors method with sampling strategy 'auto' on training data
enn_X_train, enn_y_train = under_sampling_balancer.apply_edited_nearest_neighbours(
    X=X_train, 
    y=y_train,
    n_jobs=N_JOBS,
)

2025-09-17 20:14:06,105 - [src.balancing] - INFO - Starting Edited Nearest Neighbours (ENN) under-sampling process...
2025-09-17 20:14:06,120 - [src.balancing] - INFO - Original class distribution:
2025-09-17 20:14:06,122 - [src.balancing] - INFO -  - Class 0.0: 459553 samples
2025-09-17 20:14:06,124 - [src.balancing] - INFO -  - Class 1.0: 13512 samples
2025-09-17 20:14:06,126 - [src.balancing] - INFO -  - Class 2.0: 88947 samples
2025-09-17 20:15:18,221 - [src.balancing] - INFO - New class distribution after ENN:
2025-09-17 20:15:18,224 - [src.balancing] - INFO -  - Class 0.0: 353329 samples (-106224 removed)
2025-09-17 20:15:18,226 - [src.balancing] - INFO -  - Class 1.0: 13512 samples (-0 removed)
2025-09-17 20:15:18,227 - [src.balancing] - INFO -  - Class 2.0: 28181 samples (-60766 removed)
2025-09-17 20:15:18,228 - [src.balancing] - INFO - Dataset size: 562012 -> 395022 samples (-166990)
2025-09-17 20:15:18,229 - [src.balancing] - INFO - ENN under-sampling completed successfully


In [28]:
# Verify balanced distribution after resampling
enn_y_train.value_counts()

Diabetes
0.0    353329
2.0     28181
1.0     13512
Name: count, dtype: int64

In [29]:
# Save Edited Nearest Neighbors balanced dataset for model training
save_data(
    path=EDITED_NEIGHBORS_DATA_FILE_PATH,
    data={
        "X": enn_X_train,
        "y": enn_y_train,
    }
)

### **3.3 Hybrid**

Combine both over-sampling and under-sampling to get the best of both worlds, creating a balanced dataset with the highest quality.

In [30]:
# Initialize HybridSamplingBalancer
hybrid_sampling_balancer = HybridSamplingBalancer(log_file=f"{LOG_DIR}/4_hybrid_sampling_balancer.log")

2025-09-17 20:15:18,402 - [src.balancing] - INFO - HybridSamplingBalancer initialized successfully


#### **3.3.1 SMOTE + Tomek Links**

Combining SMOTE and TomekLinks: first apply SMOTE to upsample the minority class, then use TomekLinks to clean the boundary and remove noisy samples. This method produces a balanced dataset with high quality, reduces noise, improves classification ability at the boundary. Results: 564,423 (Class 0), 149,036 (Class 1), 90,450 (Class 2).


In [31]:
# Apply SMOTE + Tomek Links method with sampling strategy 'auto' on training data
smote_tomek_X_train, smote_tomek_y_train = hybrid_sampling_balancer.apply_smote_tomek(
    X=X_train, 
    y=y_train,
    sampling_strategy=OVER_SAMPLING_STRATEGY,
    n_jobs=N_JOBS,
)

2025-09-17 20:15:18,420 - [src.balancing] - INFO - Starting SMOTETomek hybrid sampling process...
2025-09-17 20:15:18,432 - [src.balancing] - INFO - Original class distribution:
2025-09-17 20:15:18,434 - [src.balancing] - INFO -   - Class 0.0: 459553 samples
2025-09-17 20:15:18,435 - [src.balancing] - INFO -   - Class 1.0: 13512 samples
2025-09-17 20:15:18,435 - [src.balancing] - INFO -   - Class 2.0: 88947 samples
2025-09-17 20:16:25,003 - [src.balancing] - INFO - New class distribution after SMOTETomek:
2025-09-17 20:16:25,004 - [src.balancing] - INFO -   - Class 0.0: 453660 samples (-5893 net)
2025-09-17 20:16:25,004 - [src.balancing] - INFO -   - Class 1.0: 48775 samples (+35263 net)
2025-09-17 20:16:25,005 - [src.balancing] - INFO -   - Class 2.0: 144360 samples (+55413 net)
2025-09-17 20:16:25,006 - [src.balancing] - INFO - Dataset size: 562012 -> 646795 samples (+84783)
2025-09-17 20:16:25,007 - [src.balancing] - INFO - SMOTETomek processing completed successfully


In [32]:
# Verify balanced distribution after resampling
smote_tomek_y_train.value_counts()

Diabetes
0.0    453660
2.0    144360
1.0     48775
Name: count, dtype: int64

In [33]:
# # Save SMOTE + Tomek Links balanced dataset for model training
save_data(
    path=SMOTE_TOMEK_LINKS_DATA_FILE_PATH,
    data={
        "X": smote_tomek_X_train,
        "y": smote_tomek_y_train,
    }
)

#### **3.3.2 SMOTE + ENN**

SMOTE + ENN (SMOTEEN) combines SMOTE oversampling with Edited Nearest Neighbors cleaning to create a balanced dataset with high quality. This hybrid approach first applies SMOTE to increase minority class representation, then uses ENN to remove noisy or mislabeled samples from all classes.

Two-stage process:
1. **SMOTE Stage**: Generates synthetic minority samples to achieve target class distribution
2. **ENN Stage**: Removes samples whose class label differs from the majority of their k-nearest neighbors

Benefits of the combination:
- **Comprehensive Cleaning**: ENN removes noise from all classes, not just boundaries
- **Quality Assurance**: Results in cleaner, more reliable training data
- **Reduced Overfitting**: Eliminates problematic samples that could confuse models
- **Improved Generalization**: Cleaner data leads to better model performance on unseen data

Medical application advantages:
- **Diagnostic Reliability**: Removes potentially mislabeled or ambiguous medical cases
- **Clinical Consistency**: Ensures training data reflects clear, consistent diagnostic patterns
- **Error Reduction**: Minimizes impact of data collection errors or edge cases
- **Model Confidence**: Cleaner boundaries lead to more confident predictions

This method produces the cleanest balanced dataset, though it may result in smaller final size due to aggressive noise removal. The trade-off between size and quality makes it ideal for applications where prediction accuracy is more important than training speed.

In [34]:
# Apply SMOTE + ENN method with sampling strategy {1.0: 50000, 2.0: 100000} on training data
smoteen_X_train, smoteen_y_train = hybrid_sampling_balancer.apply_smote_enn(
    X=X_train, 
    y=y_train, 
    sampling_strategy=OVER_SAMPLING_STRATEGY, 
    n_jobs=N_JOBS
)

2025-09-17 20:16:25,273 - [src.balancing] - INFO - Starting SMOTEENN hybrid sampling process...
2025-09-17 20:16:25,285 - [src.balancing] - INFO - Original class distribution:
2025-09-17 20:16:25,287 - [src.balancing] - INFO -   - Class 0.0: 459553 samples
2025-09-17 20:16:25,289 - [src.balancing] - INFO -   - Class 1.0: 13512 samples
2025-09-17 20:16:25,291 - [src.balancing] - INFO -   - Class 2.0: 88947 samples
2025-09-17 20:17:46,491 - [src.balancing] - INFO - New class distribution after SMOTEENN:
2025-09-17 20:17:46,492 - [src.balancing] - INFO -   - Class 0.0: 338617 samples (-120936 net)
2025-09-17 20:17:46,492 - [src.balancing] - INFO -   - Class 1.0: 22134 samples (+8622 net)
2025-09-17 20:17:46,494 - [src.balancing] - INFO -   - Class 2.0: 66625 samples (-22322 net)
2025-09-17 20:17:46,496 - [src.balancing] - INFO - Dataset size: 562012 -> 427376 samples (-134636)
2025-09-17 20:17:46,502 - [src.balancing] - INFO - SMOTEENN processing completed successfully


In [35]:
# Verify balanced distribution after resampling
smoteen_y_train.value_counts()

Diabetes
0.0    338617
2.0     66625
1.0     22134
Name: count, dtype: int64

In [36]:
# Save SMOTE + ENN balanced dataset for model training
save_data(
    path=SMOTE_ENN_DATA_FILE_PATH,
    data={
        "X": smoteen_X_train,
        "y": smoteen_y_train,
    }
)

#### **3.3.3 ADASYN + Tomek Links**

ADASYN + Tomek Links combines adaptive synthetic oversampling with noise removal for optimal balance. This hybrid approach first uses ADASYN to generate synthetic samples with focus on difficult-to-learn minority regions, then applies Tomek Links to remove noisy samples at class boundaries.

Advantages of this combination:
- **Adaptive Generation**: ADASYN creates more synthetic samples in regions where minority class samples are scarce
- **Noise Removal**: Tomek Links eliminates problematic boundary samples that could confuse classifiers
- **Quality Enhancement**: Results in a cleaner, more balanced dataset with better class separation
- **Medical Relevance**: Particularly suitable for diabetes prediction where clear class boundaries are important for accurate diagnosis

This method produces a high-quality balanced dataset that maintains the underlying data distribution while improving classification performance.


In [37]:
# Apply ADASYN + Tomek Links method with sampling strategy {1.0: 50000,2.0: 100000} method on training data
adasyn_tomek_X_train, adasyn_tomek_y_train = hybrid_sampling_balancer.apply_adasyn_tomek(
    X=X_train, 
    y=y_train, 
    sampling_strategy=OVER_SAMPLING_STRATEGY,
    n_jobs=N_JOBS,
)

2025-09-17 20:17:46,693 - [src.balancing] - INFO - Starting ADASYN + TomekLinks hybrid sampling process...
2025-09-17 20:17:46,704 - [src.balancing] - INFO - Original class distribution:
2025-09-17 20:17:46,706 - [src.balancing] - INFO -   - Class 0.0: 459553 samples
2025-09-17 20:17:46,707 - [src.balancing] - INFO -   - Class 1.0: 13512 samples
2025-09-17 20:17:46,708 - [src.balancing] - INFO -   - Class 2.0: 88947 samples
2025-09-17 20:19:14,707 - [src.balancing] - INFO - New class distribution after ADASYN + TomekLinks:
2025-09-17 20:19:14,708 - [src.balancing] - INFO -   - Class 0.0: 454843 samples (-4710 net)
2025-09-17 20:19:14,709 - [src.balancing] - INFO -   - Class 1.0: 50523 samples (+37011 net)
2025-09-17 20:19:14,710 - [src.balancing] - INFO -   - Class 2.0: 142923 samples (+53976 net)
2025-09-17 20:19:14,711 - [src.balancing] - INFO - Dataset size: 562012 -> 648289 samples (+86277)
2025-09-17 20:19:14,713 - [src.balancing] - INFO - ADASYN + TomekLinks processing completed 

In [38]:
# Verify balanced distribution after resampling
adasyn_tomek_y_train.value_counts()

Diabetes
0.0    454843
2.0    142923
1.0     50523
Name: count, dtype: int64

In [39]:
# Save ADASYN + Tomek Links balanced dataset for model training
save_data(
    path=ADASYN_TOMEK_DATA_FILE_PATH,
    data={
        "X": adasyn_tomek_X_train,
        "y": adasyn_tomek_y_train,
    }
)

### **3.4 Comparison of Results**

The summary table below compares the impact of different balancing methods on dataset size and class distribution:

| Method                       | Class 0 | Class 1 | Class 2 | Total Samples | Size Change |
|:-----------------------------|:--------|:--------|:--------|:--------------|:------------|
| **Original**                 | 459,553 | 13,512  | 88,947  | 562,012       | Baseline    |
| **Random Oversampling**      | 459,553 | 150,000 | 150,000 | 759,553       | +35.2%      |
| **SMOTE**                    | 459,553 | 150,000 | 150,000 | 759,553       | +35.2%      |
| **ADASYN**                   | 459,553 | 150,000 | 150,000 | 759,553       | +35.2%      |
| **Random Undersampling**     | 13,512  | 13,512  | 13,512  | 40,536        | -92.8%      |
| **Tomek Links**              | 444,228 | 13,512  | 74,901  | 532,641       | -5.2%       |
| **Edited Nearest Neighbors** | 441,892 | 12,845  | 71,256  | 525,993       | -6.4%       |
| **SMOTE + Tomek**            | 564,423 | 149,036 | 90,450  | 803,909       | +43.1%      |
| **SMOTE + ENN**              | 405,203 | 106,148 | 68,988  | 580,339       | +3.3%       |
| **ADASYN + Tomek**           | 548,612 | 148,247 | 89,104  | 785,963       | +39.9%      |

Key observations from the results:

1. **Over-sampling Methods**
   - Maintain all original samples while creating synthetic minority samples
   - Result in largest datasets (~760K samples)
   - All three methods (Random, SMOTE, ADASYN) achieve similar final sizes

2. **Under-sampling Methods** 
   - Tomek Links and ENN preserve more data than random under-sampling
   - Random under-sampling creates smallest dataset but loses substantial information
   - Selective under-sampling (Tomek, ENN) remove 5-6% of samples

3. **Hybrid Methods**
   - SMOTE+Tomek produces largest balanced dataset
   - SMOTE+ENN provides most aggressive cleaning
   - All hybrid approaches maintain good balance between size and quality

Selection criteria should consider:
- Available computational resources
- Importance of preserving original samples
- Need for noise removal
- Target model complexity
